# 01 — Nguồn OHLCV và kiểm định

Dự báo log return phiên kế tiếp, sau đó suy ra giá và hướng. Chạy các ô từ trên xuống. Dữ liệu dừng tại 30/09/2025.

## Chuẩn bị môi trường

Trên máy cá nhân, cài `pip install -r requirements.txt` trước. Các lệnh mặc định dùng snapshot được lưu trong dự án; không tự thay dữ liệu bằng lần tải mạng mới.

In [ ]:
# Tìm thư mục gốc khi chạy từ repo, từ notebooks/ hoặc trên Google Colab.
from pathlib import Path
import sys, subprocess, json, os
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/fpt-stock-trends-prediction')]
ROOT = next((p for p in candidates if (p/'src/train.py').exists()), None)
if ROOT is None and 'google.colab' in sys.modules:
    subprocess.run(['git','clone','https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git','/content/fpt-stock-trends-prediction'], check=True)
    ROOT = Path('/content/fpt-stock-trends-prediction')
if ROOT is None:
    raise FileNotFoundError('Hãy mở notebook trong repository đã tải về.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src'))
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import pandas as pd
from IPython.display import display, Image
print('Thư mục làm việc:', ROOT)


## Sáu cột OHLCV: một dòng là một phiên giao dịch

Nguồn: **VCI/Vietcap qua vnstock**; script gọi `Quote(symbol="FPT", source="VCI").history(..., interval="1D")`. Bản lưu có **4.671 hàng lưu, 02/01/2007–30/09/2025**. OHLCV là viết tắt của giá mở cửa (Open), cao nhất (High), thấp nhất (Low), đóng cửa (Close), khối lượng (Volume).

| Cột | Ý nghĩa | Đơn vị và cách đọc |
|---|---|---|
| `time` | Ngày giao dịch FPT | `YYYY-MM-DD`, đối chiếu lại ngày đóng cửa HOSE khi chuẩn bị dữ liệu |
| `open` | Giá mở cửa | Nghìn VND/cổ phiếu |
| `high`, `low` | Giá cao nhất, thấp nhất trong phiên | Nghìn VND/cổ phiếu; mô tả biên độ |
| `close` | Giá đóng cửa | Nghìn VND/cổ phiếu; dùng tính lợi nhuận và nhãn |
| `volume` | Số cổ phiếu được giao dịch theo nguồn | Cổ phiếu, không phải giá trị giao dịch bằng tiền |

Ví dụ `close=82.54` trên thang snapshot tương ứng 82.540 VND/cổ phiếu. Giá lịch sử theo cơ sở điều chỉnh của nguồn, không mặc định bằng đúng bảng giá từng nhìn thấy trong quá khứ. Snapshot là bản chụp dữ liệu cố định, giúp cả nhóm so sánh trên cùng phiên bản.

Ô dưới kiểm tra schema, ngày, giá trị và dấu kiểm tra tệp; `--verify` không tải thêm dữ liệu. Xem [hướng dẫn nguồn dữ liệu](../data/README.md) để phân biệt tải mới, nhập CSV và kiểm tra bản lưu.

In [ ]:
from fetch_data import main as fetch
assert fetch(['--verify']) == 0
prices=pd.read_csv(ROOT/'data/raw/fpt/fpt_historical_price.csv')
display(prices.head())
display(prices.tail())

## Khi có bản giá mới

Notebook xác minh bản giá cố định. Khi có bản giá mới, kiểm tra đơn vị, ngày, cơ sở điều chỉnh và lịch HOSE trước khi chạy lại các bước. Phiên bản vnstock dùng để truy vấn bản gốc đã ngừng truy cập trực tuyến.

In [ ]:
display(pd.DataFrame({'dtype':prices.dtypes.astype(str),'missing':prices.isna().sum()}))